In [1]:
from pyspark.sql import SparkSession 

In [2]:
spark = SparkSession.builder.appName("Case Study").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 04:14:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

26/06/16 04:14:46 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [4]:
total_customers = customers.count()
total_products = products.count()
total_orders = orders.count()
total_returned_orders = returns.count()

In [5]:
print(f"Total Customers: {total_customers}")
print(f"Total Products: {total_products}")
print(f"Total Orders: {total_orders}")
print(f"Total Returned Orders: {total_returned_orders}")

Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


In [6]:
from pyspark.sql.functions import sum, col
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
sales_data = order_items.join(products, "product_id")

sales_data.groupBy("category") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_sales")
    ) \
    .show()

[Stage 23:=============================>                            (1 + 1) / 2]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [7]:
from pyspark.sql.functions import sum, col

top_customers = orders.join(order_items, "order_id") \
    .groupBy("customer_id") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_purchase")
    ) \
    .orderBy(col("total_purchase").desc()) \
    .limit(10)

top_customers.show()

[Stage 30:>                                                         (0 + 2) / 2]

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      93094|         181569.68|
|      64560|169060.39999999997|
|      23289|161573.80000000002|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|         151037.32|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|148281.03999999998|
+-----------+------------------+



In [8]:
from pyspark.sql.functions import year , month , max , sum , col

In [9]:
last_year = orders.select(
    max(year("order_date"))
).collect()[0][0]

print (last_year)

[Stage 35:=============================>                            (1 + 1) / 2]

2024


In [10]:
orders_last_year = orders.filter(
    year("order_date") == last_year
)

In [11]:
sales_df = orders_last_year.join(
    order_items,
    "order_id"
)

In [12]:
monthly_sales = sales_df.groupBy(
    month("order_date").alias("month")
).agg(
    sum(
        col("selling_price") *
        col("quantity")
    ).alias("total_sales")
).orderBy("month")

monthly_sales.show()

[Stage 42:>                                                         (0 + 2) / 2]

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1| 4.445777757600032E8|
|    2| 4.153661441999996E8|
|    3| 4.436282454099993E8|
|    4|4.2782097433999574E8|
|    5|4.4481061894999886E8|
|    6| 4.317051540600023E8|
|    7| 4.436705191199994E8|
|    8|4.4109517702000153E8|
|    9| 4.310715260799986E8|
|   10|4.4136378931000113E8|
|   11| 4.336233640400014E8|
|   12| 4.427129083499967E8|
+-----+--------------------+



In [13]:
from pyspark.sql.functions import count, col

total_orders = order_items.join(products, "product_id") \
    .groupBy("category").agg(count("order_id").alias("total_orders"))

returned_orders = order_items.join(products, "product_id") \
    .join(returns, "order_id") .groupBy("category") .agg(count("order_id").alias("returned_orders"))

result = total_orders.join(returned_orders, "category") \
    .withColumn(
        "return_percentage",
        (col("returned_orders") * 100 / col("total_orders"))
    )

result.select("category", "return_percentage").show()

+--------------+------------------+
|      category| return_percentage|
+--------------+------------------+
|Home & Kitchen|10.003363791776682|
|        Sports|10.020923065323318|
|   Electronics|10.002676709807089|
|      Clothing| 9.976450338745623|
|         Books|10.023508145900358|
|        Beauty|10.032354191296188|
|          Toys|10.079039445376354|
+--------------+------------------+



In [14]:
from pyspark.sql.functions import count, row_number
from pyspark.sql.window import Window

payment_count = orders.join(customers, "customer_id") \
    .groupBy("state", "payment_mode") \
    .count()

window = Window.partitionBy("state").orderBy(payment_count["count"].desc())

payment_count.withColumn(
    "rank",
    row_number().over(window)
).filter("rank=1").show()

[Stage 56:=============================>                            (1 + 1) / 2]

+-----+----------------+-----+----+
|state|    payment_mode|count|rank|
+-----+----------------+-----+----+
|   CA|             UPI|20246|   1|
|   FL|      Debit Card|20010|   1|
|   GA|     Net Banking|20041|   1|
|   IL|Cash on Delivery|20498|   1|
|   MI|      Debit Card|20416|   1|
|   NC|     Net Banking|19596|   1|
|   NY|      Debit Card|20369|   1|
|   OH|     Net Banking|20351|   1|
|   TX|             UPI|20065|   1|
|   WA|             UPI|20244|   1|
+-----+----------------+-----+----+



In [16]:
from pyspark.sql.functions import countDistinct, sum, col

result = orders.join(order_items, "order_id") \
    .join(products, "product_id") \
    .groupBy("customer_id") \
    .agg(
        countDistinct("category").alias("category_count"),
        sum(col("selling_price") * col("quantity")).alias("total_spent")
    ) \
    .filter(
        (col("category_count") >= 5) &
        (col("total_spent") > 100000)
    )

result.show()

26/06/16 04:31:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:31:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 71:>                                                         (0 + 2) / 2]

+-----------+--------------+------------------+
|customer_id|category_count|       total_spent|
+-----------+--------------+------------------+
|      82672|             7|         104905.22|
|      64423|             7|122832.84000000001|
|      91446|             7|         122214.13|
|      91367|             7|          107376.9|
|      57984|             7|127489.03999999998|
|       9376|             7|107470.09999999999|
|      85749|             7|         109065.03|
|      81900|             7|         127465.68|
|      85100|             7|101890.51999999999|
|       2659|             7|         116119.88|
|      60965|             7|         104585.06|
|      22990|             7|115511.68999999999|
|      43588|             7|111164.06999999999|
|      68433|             7|108316.26999999999|
|      80943|             7|118172.32999999999|
|      68544|             7|         115948.79|
|      35450|             7|109224.42000000001|
|      36655|             7|         104